# SAMap supplementary figure, panel d: cross-species edges link matching compartments

How the SAMap alignment connects the two species, before any value is transferred. For the nuclei of one
compartment and species, each bar splits their summed cross-species edge weight by the compartment of the other
species it lands in; the number is the share landing in the matching compartment (methods: 88, 82, 85 and 92%).

**Input**: `samap_compartment_edge_share.csv`, written by rule `samap_edge_share` (`samap_edge_share.py`): one row
per direction (`human_to_mouse`, `mouse_to_human`) and source compartment, one column per target compartment.
**Outputs**, in `OUT`: `sfig_samap_panel_d.png` (300 dpi), `sfig_samap_panel_d_600dpi.png`,
`sfig_samap_panel_d.svg` (text kept as text) and `sfig_samap_panel_d.provenance.json`.
`sfig_samap_compose.ipynb` places the panel in the figure and adds its letter.

**Running it.** Run all cells with this directory as the working directory: the shared modules next to the notebook
are imported from there. The cells above *Run* only define constants and functions. Paths are relative to the
repository root. Existing outputs are not overwritten unless `FORCE = True`.

The pipeline runs it with papermill (`workflow/rules/10c_samap_figures.smk`). By hand, from the repository root:

```bash
papermill notebooks/samap/sfig_samap_d_edge_share.ipynb \
    results/notebooks/samap/sfig_samap_panel_d.executed.ipynb \
    --cwd notebooks/samap -p FORCE True
```

Commit this notebook without outputs: the provenance record notes whether the notebook matches its commit.

In [ ]:
# Parameters. papermill -p NAME VALUE overrides them in a cell below this one.
# Paths are relative to the repository root; REPO = None finds the root with git from the working directory.
REPO = None
EDGE_SHARE = "results/samap_figures/edge_share/samap_compartment_edge_share.csv"
OUT = "results/figures/samap_supp/panels"
FORCE = False  # overwrite existing outputs
NOTEBOOK = "notebooks/samap/sfig_samap_d_edge_share.ipynb"  # this notebook, for the provenance record

In [ ]:
import logging
import sys

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import FancyArrowPatch, Patch
from IPython.display import Image, display

import sfig_samap_style as style

# This notebook's messages in the cell output; other libraries stay at WARNING.
logging.basicConfig(format="%(message)s", stream=sys.stdout, force=True)
LOG = logging.getLogger("sfig_samap_d")
LOG.setLevel(logging.INFO)

## Constants

In [ ]:
LETTER = "d"
TITLE = "Cross-species edges link\nmatching compartments"
WIDTH_SHOWN = 300  # px, the PNG shown in this notebook

# One bar per direction and source compartment, top to bottom; the gap at 1.8 separates the two directions.
BARS = [
    ("human_to_mouse", "endometrial_fibroblast", "fibroblast"),
    ("human_to_mouse", "decidual_cell", "decidual"),
    ("mouse_to_human", "endometrial_fibroblast", "fibroblast"),
    ("mouse_to_human", "decidual_cell", "decidual"),
]
BAR_Y = [3.6, 2.6, 1.0, 0.0]
DIRECTION_LABEL_Y = {"human_to_mouse": 4.15, "mouse_to_human": 1.55}  # data units, above each pair of bars
LABEL_GREY = "#555555"

## Figure

In [ ]:
def direction_label(fig, ax, width_mm, y, source, target):
    """'source -> target' above a pair of bars, the arrow drawn as a line.

    Helvetica Neue has no arrow glyph, and an SVG that leaves the arrow to a fallback font shows a missing glyph
    wherever that font is not installed.
    """
    text_style = dict(xycoords=("figure fraction", "data"), ha="left", va="bottom", fontsize=8, color=LABEL_GREY)
    first = ax.annotate(source, xy=(style.TITLE_INSET_MM / width_mm, y), **text_style)
    extent = first.get_window_extent(fig.canvas.get_renderer())
    (_, y0), (x1, y1) = fig.transFigure.inverted().transform(extent)
    y_mid = y0 + 0.42 * (y1 - y0)  # middle of the x-height
    gap, length = 0.9 / width_mm, 3.4 / width_mm
    fig.add_artist(FancyArrowPatch(
        (x1 + gap, y_mid),
        (x1 + gap + length, y_mid),
        transform=fig.transFigure,
        arrowstyle="-|>",
        mutation_scale=5,
        lw=0.7,
        color=LABEL_GREY,
        shrinkA=0,
        shrinkB=0,
    ))
    ax.annotate(target, xy=(x1 + 2 * gap + length, y), **text_style)


def draw_edge_share(edge_share: pd.DataFrame):
    """100% stacked bars of edge weight by target compartment, the matching share printed in its segment."""
    fig, width_mm, height_mm = style.new_panel(LETTER)
    ax = fig.add_axes([18 / width_mm, 11 / height_mm, 1 - 21 / width_mm, 1 - 28 / height_mm])

    for y, (direction, source, _) in zip(BAR_Y, BARS):
        to_fibroblast = edge_share.loc[(direction, source), "endometrial_fibroblast"]
        to_decidual = edge_share.loc[(direction, source), "decidual_cell"]
        ax.barh(y, 100 * to_fibroblast, color=style.FIBROBLAST, height=0.72, linewidth=0)
        ax.barh(y, 100 * to_decidual, left=100 * to_fibroblast, color=style.DECIDUAL, height=0.72, linewidth=0)

        # The matching share, centered in its own segment, near-black on either color.
        is_fibroblast = source == "endometrial_fibroblast"
        match = to_fibroblast if is_fibroblast else to_decidual
        x_center = 100 * to_fibroblast / 2 if is_fibroblast else 100 * (to_fibroblast + to_decidual / 2)
        ax.text(x_center, y, f"{100 * match:.0f}%", ha="center", va="center", fontsize=8,
                color="#222222")

    ax.set_yticks(BAR_Y)
    ax.set_yticklabels([bar[2] for bar in BARS])
    ax.tick_params(axis="y", length=0)
    ax.set_xlim(0, 100)
    ax.set_xticks([0, 50, 100])
    ax.set_ylim(-0.6, 4.75)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel("share of edge weight (%)")

    direction_label(fig, ax, width_mm, DIRECTION_LABEL_Y["human_to_mouse"], "human", "mouse")
    direction_label(fig, ax, width_mm, DIRECTION_LABEL_Y["mouse_to_human"], "mouse", "human")
    ax.legend(
        handles=[Patch(color=style.FIBROBLAST, label="to fibroblast"),
                 Patch(color=style.DECIDUAL, label="to decidual")],
        loc="upper left",
        bbox_to_anchor=(style.TITLE_INSET_MM / width_mm, 1 - 10.5 / height_mm),
        bbox_transform=fig.transFigure,
        ncol=2,
        frameon=False,
        fontsize=8,
        handlelength=0.9,
        handletextpad=0.3,
        columnspacing=1.0,
        borderaxespad=0,
    )
    style.panel_title(fig, width_mm, height_mm, TITLE)
    return fig

## Run

In [ ]:
repo = style.find_repo(REPO)
notebook = repo / NOTEBOOK
if not notebook.is_file():
    raise FileNotFoundError(f"{notebook} not found; set NOTEBOOK to this notebook's path")

paths = style.panel_paths(repo / OUT, LETTER)
style.refuse_overwrite(paths.values(), FORCE)
paths["png"].parent.mkdir(parents=True, exist_ok=True)

In [ ]:
edge_share = pd.read_csv(repo / EDGE_SHARE, index_col=[0, 1])

# The shares printed in the bars: edge weight landing in the matching compartment.
matching_share = {
    f"{direction}/{source}": round(float(edge_share.loc[(direction, source), source]), 4)
    for direction, source, _ in BARS
}
LOG.info("Matching share: %s", matching_share)
display(edge_share)

In [ ]:
font = style.choose_font()
style.apply_style(font)
LOG.info("Font: %s", font)

fig = draw_edge_share(edge_share)

style.save_panel(fig, LETTER, paths)
plt.close(fig)
LOG.info("Wrote %s", [str(path) for name, path in paths.items() if name != "provenance"])

# The saved PNG rather than the live figure, so what shows here is the file itself.
display(Image(filename=str(paths["png"]), width=WIDTH_SHOWN))

In [ ]:
record = style.provenance(
    notebook,
    parameters={"REPO": str(repo), "EDGE_SHARE": EDGE_SHARE, "OUT": OUT, "FORCE": FORCE},
    inputs={"edge_share": repo / EDGE_SHARE},
    outputs=[paths["png"], paths["png_600dpi"], paths["svg"]],
    font=font,
)
record["panel"] = style.panel_spec(LETTER)
record["matching_share"] = matching_share
style.write_json(paths["provenance"], record)
LOG.info("Wrote %s", paths["provenance"])